# Portfolio risk and causal VaR forecasts

公开整理的教学研究 Notebook：组合约束、风险导数、因果预测与共同样本评价。
本版本重新组织了代码及实验设计，不包含考试题目、提交报告、行情或历史输出。

默认使用合成数据，仅验证流程；任何输出都不是市场发现或投资结论。输入合同、修改说明和统计限制见本目录上一级 README。

In [ ]:
from pathlib import Path
import importlib.util
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
elif not (project_dir / "analysis.py").is_file():
    project_dir = project_dir / "projects" / "risk-model-validation"
spec = importlib.util.spec_from_file_location("risk", project_dir / "analysis.py")
risk = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = risk
spec.loader.exec_module(risk)

## 1. A fully invested target-return portfolio

以下预期收益、波动率和相关系数均是人为设定，使用同一抽象时间尺度。
组合允许卖空；没有加杠杆上限或交易成本约束。约束为权重和等于 1、预期收益等于目标值。
正态 VaR/ES 使用正损失口径；权重偏导是其他权重固定时的局部导数。

In [ ]:
mu = np.array([0.025, 0.055, 0.09, 0.12])
vol = np.array([0.06, 0.11, 0.16, 0.22])
correlation = np.full((4, 4), 0.25)
np.fill_diagonal(correlation, 1.0)
covariance = np.outer(vol, vol) * correlation
target = 0.065
weights = risk.minimum_variance_for_target(mu, covariance, target)
print("Weight sum:", weights.sum(), "target return:", weights @ mu)
display(pd.DataFrame({"weight": weights, "assumed_mean": mu}))
portfolio = risk.normal_loss_risk(weights, mu, covariance)
print("Normal loss VaR:", portfolio["var"], "ES:", portfolio["es"])
display(pd.DataFrame({"dVaR_dw": portfolio["dvar_dw"], "dES_dw": portfolio["des_dw"]}))

## 2. Local inputs or a synthetic demonstration

`DATA_PATH = None` 产生固定种子的合成价格，不保存文件。
如使用自己的行情，显式指定仓库外 CSV，包含 `date,close`，日期严格递增、无重复，价格有限且为正。
调用者负责交易日完整性和价格口径；10 期指后续 10 行，不是 10 个自然日。

In [ ]:
DATA_PATH = None  # Example: Path("/path/outside/repository/prices.csv")
WINDOW = 21
HORIZON = 10
CONFIDENCE = 0.99
DECAY = 0.72
prices = risk.synthetic_prices() if DATA_PATH is None else risk.load_prices(DATA_PATH)
print("SYNTHETIC DEMO ONLY" if DATA_PATH is None else "User-supplied local data")
print("Observations:", len(prices))
print("EWMA new squared-return weight:", 1 - DECAY)
print("EWMA half-life in observations:", np.log(0.5) / np.log(DECAY))

## 3. Same origins and information cutoff

每个预测在 t 日收盘后产生，三个方法都使用截至 t 的可知信息。
滚动法、经验分位数取同一个包含 r[t] 的窗口；EWMA 用最初窗口初始化后仅向前递推。
未来损失只用于评价。三个模型共用同一组有效预测日期，不因起始窗口不同而改变统计样本。

经验日收益分位数乘以平方根时间只是近似，21 个观测估计 1% 尾部很不稳定；不据此预设模型排名。

In [ ]:
forecasts = risk.risk_forecasts(prices, window=WINDOW, horizon=HORIZON,
                                 confidence=CONFIDENCE, decay=DECAY)
common = risk.common_backtest(forecasts)
display(risk.coverage_summary(common, confidence=CONFIDENCE))
risk.plot_backtest(common)
plt.show()

## 4. Coverage diagnostics and their limits

上表的逐日 10 期损失区间重叠，只作描述统计。
下表每隔 10 行抽取一次，去除收益区间重叠；但并不保证超限独立。
Kupiec 的 p 值、二项分布色带都只是 iid 假设下的参考，不能解释为监管交通灯或“模型通过”。
不拒绝原假设不等于证明模型正确，低样本量还会影响检验效力。

In [ ]:
display(risk.coverage_summary(common, confidence=CONFIDENCE,
                              stride=HORIZON, iid_reference=True))

## Reproducibility and scope

从项目目录运行 `python checks.py` 可验证约束、导数、时间因果性及统计边界值。
本公开版本修正了全样本 EWMA 初始化、比较日期和符号口径，未重跑原市场数据。
当前输出来自本版本和当前输入，不应与原报告数字混合。
后续可研究更长尾部窗口、不同起始偏移、条件覆盖检验与更稳健的多期分布建模。